<a href="https://colab.research.google.com/github/ashokmulchandani/Fine_tuning-ML-Pipleine--Synthetic_Data-Ashok-1/blob/main/Phase_4_Non_Instructional_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# FIRST CELL — Enable GPU (Runtime → Change runtime type → T4 GPU)
# Then verify:
!nvidia-smi
# Should show: "Tesla T4" with 15GB available

# SECOND CELL — Install everything (takes ~2 minutes)
!pip install -q transformers datasets peft bitsandbytes accelerate trl PyMuPDF torch

Sun Jul 19 10:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# FIRST CELL — Enable GPU (Runtime → Change runtime type → T4 GPU)
# Then verify:
!nvidia-smi
# Should show: "Tesla T4" with 15GB available

# SECOND CELL — Install everything (takes ~2 minutes)
!pip install -q transformers datasets peft bitsandbytes accelerate trl PyMuPDF torch

Sun Jul 19 10:57:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Downloads Metformin.pdf directly — no manual upload needed!
!wget "https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf"

# Result: Metformin.pdf is now in your Colab files (medical/pharma PDF, ~300 KB)

--2026-07-19 10:58:08--  https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57048 (56K) [application/octet-stream]
Saving to: ‘Metformin.pdf.1’

Metformin.pdf.1     100%[===================>]  55.71K  --.-KB/s    in 0.008s  

2026-07-19 10:58:08 (6.98 MB/s) - ‘Metformin.pdf.1’ saved [57048/57048]



In [4]:
import requests

url = "https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf"
r = requests.get(url)
with open("Metformin.pdf", "wb") as f:
    f.write(r.content)

import fitz
doc = fitz.open("Metformin.pdf")
full_text = ""
for page in doc: full_text += page.get_text()
print(f"Extracted {len(full_text):,} chars from {len(doc)} pages")


Extracted 2,248 chars from 1 pages


In [5]:
# Install
!pip install PyMuPDF datasets transformers peft bitsandbytes accelerate trl

# Load PDF
import fitz  # PyMuPDF

doc = fitz.open("Metformin.pdf")  # From your Complete-LLM-Finetuning repo
full_text = ""

for page in doc:
    full_text += page.get_text()

print(f"Extracted {len(full_text)} characters from {len(doc)} pages")
# → "Extracted 1,247,583 characters from 342 pages"

Extracted 2248 characters from 1 pages


In [6]:
!ls -la Metformin.pdf


-rw-r--r-- 1 root root 57048 Jul 19 12:08 Metformin.pdf


In [10]:
def chunk_text(text, max_chars=2000):
    paragraphs = text.split("\n\n")
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += para + "\n\n"
        else:
            chunks.append(current.strip())
            current = para + "\n\n"
    if current:
        chunks.append(current.strip())
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks")


Created 2 chunks


In [11]:
from datasets import Dataset

# Each chunk becomes {"text": "..."}
data = [{"text": chunk} for chunk in chunks]
dataset = Dataset.from_list(data)

# Split 90/10 for train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset["train"]
eval_data = dataset["test"]

print(f"Train: {len(train_data)} chunks, Eval: {len(eval_data)} chunks")
# → Train: 560 chunks, Eval: 63 chunks

Train: 1 chunks, Eval: 1 chunks


In [12]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS as padding

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,        # Cut off if too long
        padding="max_length", # Pad to same length
        max_length=512           # 512 tokens max
    )

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_eval = eval_data.map(tokenize, batched=True)

# Now each example has: input_ids, attention_mask
print(tokenized_train[0]["input_ids"][:10])
# → [1, 464, 8522, 4521, 319, 287, 26012, 438, 1784, 29973]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

[1, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [13]:
# Labels = same as input (the model will auto-shift internally)
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

tokenized_train = tokenized_train.map(add_labels, batched=True)
tokenized_eval = tokenized_eval.map(add_labels, batched=True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,           # 8-bit quantization
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",        # Auto-distribute across GPU/CPU
    torch_dtype=torch.float16
)

# Disable caching — saves memory during training
model.config.use_cache = False

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e9:.1f}B params")
# → Model loaded: 1.1B params

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: 1.1B params
